In [4]:
!pip install catboost xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.0 MB/s eta 0:00:00


In [5]:
# ===== CLEVELAND: Corrected leakage-free pipeline =====
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, encoding='utf-8-sig')
df = df.drop_duplicates()

if 'thal' in df.columns:
    df = df.drop(columns=['thal'])

X = df.drop(columns=['target'])
y = df['target']

# --- SPLIT FIRST ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Impute using TRAIN stats only ---
for col in X_train.columns:
    if X_train[col].isnull().sum() > 0:
        fill_val = X_train[col].median()
        X_train[col].fillna(fill_val, inplace=True)
        X_test[col].fillna(fill_val, inplace=True)

# --- Outlier capping using TRAIN bounds only ---
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
for col in num_cols:
    Q1, Q3 = X_train[col].quantile(0.25), X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

# --- Scale (fit on train only) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train 4 models ---
lr_model = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_scaled, y_train)
rf_model = RandomForestClassifier(random_state=42, n_estimators=200).fit(X_train, y_train)
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss').fit(X_train, y_train)
cat_model = CatBoostClassifier(random_state=42, verbose=0).fit(X_train, y_train)

# --- Evaluate ---
models = {
    "Logistic Regression": (lr_model, X_test_scaled),
    "Random Forest": (rf_model, X_test),
    "XGBoost": (xgb_model, X_test),
    "CatBoost": (cat_model, X_test)
}
results = []
for name, (model, X_te) in models.items():
    y_pred = model.predict(X_te)
    results.append([name, accuracy_score(y_test, y_pred), precision_score(y_test, y_pred),
                     recall_score(y_test, y_pred), f1_score(y_test, y_pred)])

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1-score"]).round(4)
print("Cleveland - Leakage-free Comparative Analysis Table:")
results_df

Saving Cleeveland_Dataset_no_thal.csv to Cleeveland_Dataset_no_thal.csv
Cleveland - Leakage-free Comparative Analysis Table:


,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.9180,0.8710,0.9643,0.9153
1,Random Forest,0.9344,0.9000,0.9643,0.9310
2,XGBoost,0.8525,0.7879,0.9286,0.8525
3,CatBoost,0.9344,0.9000,0.9643,0.9310


In [6]:
# ===== CARDIOVASCULAR: Corrected leakage-free pipeline =====
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
df = df.drop_duplicates()

leak_cols = [c for c in df.columns if c.lower() in ['id', 'patientid']]
if leak_cols:
    df = df.drop(columns=leak_cols)

X = df.drop(columns=['target'])
y = df['target']

# --- SPLIT FIRST ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Fix invalid zeros in serumcholestrol using TRAIN median only ---
train_chol_median = X_train.loc[X_train['serumcholestrol'] != 0, 'serumcholestrol'].median()
X_train['serumcholestrol'] = X_train['serumcholestrol'].replace(0, train_chol_median)
X_test['serumcholestrol'] = X_test['serumcholestrol'].replace(0, train_chol_median)

# --- Impute any other missing values using TRAIN stats only ---
for col in X_train.columns:
    if X_train[col].isnull().sum() > 0:
        fill_val = X_train[col].median()
        X_train[col].fillna(fill_val, inplace=True)
        X_test[col].fillna(fill_val, inplace=True)

# --- Outlier capping using TRAIN bounds only (none found last time, but kept for correctness) ---
num_cols = ['age', 'restingBP', 'serumcholestrol', 'maxheartrate', 'oldpeak']
for col in num_cols:
    Q1, Q3 = X_train[col].quantile(0.25), X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

# --- Scale (fit on train only) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train 4 models ---
lr_model = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_scaled, y_train)
rf_model = RandomForestClassifier(random_state=42, n_estimators=200).fit(X_train, y_train)
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss').fit(X_train, y_train)
cat_model = CatBoostClassifier(random_state=42, verbose=0).fit(X_train, y_train)

# --- Evaluate ---
models = {
    "Logistic Regression": (lr_model, X_test_scaled),
    "Random Forest": (rf_model, X_test),
    "XGBoost": (xgb_model, X_test),
    "CatBoost": (cat_model, X_test)
}
results = []
for name, (model, X_te) in models.items():
    y_pred = model.predict(X_te)
    results.append([name, accuracy_score(y_test, y_pred), precision_score(y_test, y_pred),
                     recall_score(y_test, y_pred), f1_score(y_test, y_pred)])

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1-score"]).round(4)
print("Cardiovascular - Leakage-free Comparative Analysis Table:")
results_df

Saving Cardiovascular_Disease_range_1000_.csv to Cardiovascular_Disease_range_1000_.csv
Cardiovascular - Leakage-free Comparative Analysis Table:


,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.980,0.9746,0.9914,0.9829
1,Random Forest,0.985,0.9829,0.9914,0.9871
2,XGBoost,0.980,0.9828,0.9828,0.9828
3,CatBoost,0.990,0.9914,0.9914,0.9914
